# 02 数据预处理管道（02_clean.ipynb）

> 依据 `docs/03-data-preprocessing-plan.md` 实现，步骤编号 P1~P11 与方案 §2 管道总览一一对应。
> 输入：`dataset/comments.csv`（UTF-8，405,083 行）+ `dataset/movies.csv`（GBK，9,372 行）；
> 输出：`output/cleaned_comments_v2.csv`（预计约 38 万条）+ `output/clean_report.json`。
>
> 三条主线对应方案章节：**去掉没用的数据**（§3）、**修正不好的数据**（§4）、**补充不够的数据**（§5）。
>
> 运行依赖：
> - 必需：`pandas`、`numpy`（本仓库 `.venv` 已有）；
> - 可选：`opencc-python-reimplemented`（繁→简转换；未安装时只打 `IS_TRAD` 标记、不转换，管道照常运行）。
>
> 刻意**不在**本管道做的事（防泄漏，见方案 §4.6/§4.7/§5.2）：
> 分词/停用词（仅 baseline 路线、须在训练集上固化 `stopwords.txt`）；`USER_MEAN_RATING` / `MOVIE_BAYES_MEAN`（须只用训练集统计，建模阶段切分后生成）。

## 处理管道总览（方案 §2）

| 步骤 | 动作 | 类型 |
|---|---|---|
| P1 | movies.csv 按 ID 去重（修 join 扇出）+ join 键类型统一 | 修正 |
| P2 | TIME 解析，失败剔除 | 修正 |
| P3 | 剔除目标变量不可用（RATING NaN/越界） | 去掉 |
| P4 | 剔除文本不可用（CONTENT 空/纯标点）+ ID 缺失 | 去掉 |
| P5 | 剔除完全重复行 | 去掉 |
| P6 | (CREATOR, MOVIEID) 去重，保留最新（CREATOR 缺失行除外） | 去掉 |
| P7 | 剔除疑似机器人用户（阈值驱动） | 去掉 |
| P7b | TIME 越界剔除（其余异常值只标记） | 修正 |
| P8 | NLP 专有文本清洗（不改行数） | 修正 |
| P9/P10 | 标记列 + 派生特征 | 补充 |
| P11 | 输出 v2 数据集 + 清洗报告 | - |

In [2]:
import json
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.unicode.east_asian_width', True)

BASE = Path.cwd()
DATA = BASE / 'dataset'
OUT = BASE / 'output'
OUT.mkdir(exist_ok=True)

# ====== 管道参数（消融实验时改这里，方案 §1 总原则 2）======
PARAMS = {
    'SNAPSHOT_DATE': '2018-01-23 23:59:59',  # 爬取快照日（EDA §1.1：ADD_TIME 仅 1,572 个唯一值）
    'TIME_MIN': '2005-01-01',                # 豆瓣上线年，更早为脏数据
    'MAX_LEN': 512,                          # 长评截断（覆盖 RoBERTa 128 token 绰绰有余）
    'ROBOT_MIN_COMMENTS': 50,                # 疑似机器人：评论数阈值（§3.4）
    'ROBOT_MAX_MEAN_LEN': 10,                # 疑似机器人：平均长度阈值（§3.4）
}

# 清洗报告（§1 总原则 1：每步可量化、可回溯，落盘 clean_report.json）
report = {'params': PARAMS, 'steps': {}}

def step(name, before, after, note=''):
    """记录每步 before -> after 行数，保证行数守恒可回溯"""
    report['steps'][name] = {'before': int(before), 'after': int(after),
                             'removed': int(before - after), 'note': note}
    print(f'[{name:>4}] {before:>7} -> {after:>7}（剔除 {before - after:>6} 条）{note}')

print('pandas', pd.__version__)

pandas 2.3.3


## P1 加载数据 + movies.ID 去重（§4.1 / §4.2，必做）

- movies.csv 为 **GBK** 编码，读取失败回退 `gb18030`，抽样 NAME 目检乱码；
- join 键类型统一：comments 的 MOVIEID 是字符串、movies.ID 是 int64（EDA 复现过的坑）；
- **修 EDA 隐藏 bug**：EDA 里 merge 后 405,083 → 405,863（+780 行），原因是 movies.ID 有重复，left join 把部分评论复制了多份、重复参与训练。先 `drop_duplicates(subset='ID')` 再 join，并断言行数不变；
- **重要修正（§3.1）**：匹配不到电影的评论（约 26%）**不再剔除**，打 `HAS_META` 标记保留——建模输入是「CONTENT → RATING」，电影名只是展示性元数据，一刀切丢掉 10 万+ 条带标签样本代价远大于收益。

In [3]:
# ---- P1: 加载数据 + movies.ID 去重 + join 键类型统一（§4.1 / §4.2）----
comments_raw = pd.read_csv(DATA / 'comments.csv', encoding='utf-8')
try:
    movies = pd.read_csv(DATA / 'movies.csv', encoding='gbk', usecols=['ID', 'NAME'])
    enc_used = 'gbk'
except UnicodeDecodeError:
    movies = pd.read_csv(DATA / 'movies.csv', encoding='gb18030', usecols=['ID', 'NAME'])
    enc_used = 'gb18030'
print(f'comments: {comments_raw.shape} | movies: {movies.shape}（编码 {enc_used}）')

# join/排序键类型统一（EDA 已复现的坑）：MOVIEID 被读成字符串、movies.ID 是 int64；
# ID 列混有非数字字符串 -> object 混合类型（int/str 无法排序），统一 to_numeric，
# 脏 ID 变成 NaN，由 P4 统一剔除（对应方案 §3.7 的 11 条 ID 缺失）
comments_raw['ID'] = pd.to_numeric(comments_raw['ID'], errors='coerce').astype('Int64')
comments_raw['MOVIEID'] = pd.to_numeric(comments_raw['MOVIEID'], errors='coerce').astype('Int64')
movies['ID'] = pd.to_numeric(movies['ID'], errors='coerce').astype('Int64')

# §4.1 断言 NAME 无乱码：抽样 10 行目检
print('\nmovies NAME 抽查（目检无乱码）:')
display(movies['NAME'].dropna().sample(10, random_state=42).to_frame())

# §4.2 修 EDA 隐藏 bug：movies.ID 重复导致 merge 后 405,083 -> 405,863（+780 行扇出），先去重再 join
n_dup_id = int(movies['ID'].duplicated().sum())
movies_dedup = movies.drop_duplicates(subset='ID', keep='first')
report['movies_id_dup'] = n_dup_id
report['comments_bad_id'] = int(comments_raw['ID'].isna().sum())
print(f'movies.ID 重复 {n_dup_id} 条 -> 去重后 {len(movies_dedup)} 行 | comments 脏 ID（无法解析）{report["comments_bad_id"]} 条')

df = comments_raw.merge(movies_dedup, left_on='MOVIEID', right_on='ID', how='left', suffixes=('', '_m'))
assert len(df) == len(comments_raw), 'join 扇出未修复！'
df = df.rename(columns={'NAME': 'MOVIE_NAME'})
df['HAS_META'] = df['MOVIE_NAME'].notna()   # §3.1 匹配不到电影的评论保留、打标记，不剔除
print(f'join 后 {len(df)} 行 == 原始 {len(comments_raw)} 行，无扇出 ✓ '
      f'| HAS_META=False（无电影名）{int((~df["HAS_META"]).sum())} 条（约 26%，建模保留）')

comments: (405083, 7) | movies: (9372, 2)（编码 gbk）

movies NAME 抽查（目检无乱码）:


,NAME
6211,神鞭
8367,芳心谋杀案
5456,指甲刀人魔
4250,投靠女与出走男
7758,人造风景
2164,尼斯湖怪：深水传说
1142,断头谷
2534,入侵脑细胞
2018,绿茶
9146,里奥纳德·科恩：我是你的男人


movies.ID 重复 2 条 -> 去重后 9370 行 | comments 脏 ID（无法解析）111 条
join 后 405083 行 == 原始 405083 行，无扇出 ✓ | HAS_META=False（无电影名）107456 条（约 26%，建模保留）


## P2 TIME 解析（§4.1，必做）

`2015/6/10 2:45` 字符串 → datetime；解析失败（含 TIME 本身缺失约 54 条）剔除——P6 的「保留最新」和 YEAR 切分都依赖 TIME。MOVIEID 类型统一已在 P1 完成。

In [4]:
# ---- P2: TIME 解析（§4.1）----
n0 = len(df)
df['TIME_DT'] = pd.to_datetime(df['TIME'], format='%Y/%m/%d %H:%M', errors='coerce')
bad_time = df['TIME_DT'].isna()
report['time_parse_failed'] = int(bad_time.sum())
step('P2', n0, n0 - int(bad_time.sum()),
     f'TIME 缺失/无法解析 {int(bad_time.sum())} 条，剔除（去重「保留最新」与 YEAR 切分都依赖 TIME）')
df = df.loc[~bad_time].copy()

[  P2]  405083 ->  404972（剔除    111 条）TIME 缺失/无法解析 111 条，剔除（去重「保留最新」与 YEAR 切分都依赖 TIME）


## P3/P4 目标变量与文本不可用的记录（§3.2 / §3.7，必做）

- **RATING** 为 NaN 或不在 [1,5]：目标变量缺失无法补救，**绝不插补**（插补 = 造标签）；
- **CONTENT** 为空、纯空白、**纯标点/纯符号**（`^[\W_]+$`，EDA 未单独统计，管道中补计数）；
- **ID** 缺失：行主键无法追溯，剔除。

In [5]:
# ---- P3: 目标变量不可用（§3.2）----
n0 = len(df)
rating_num = pd.to_numeric(df['RATING'], errors='coerce')
bad_rating = ~rating_num.between(1, 5)      # NaN 或不在 [1,5]
step('P3', n0, n0 - int(bad_rating.sum()), 'RATING 为 NaN 或不在 [1,5]，目标变量缺失不可插补')
df = df.loc[~bad_rating].copy()
df['RATING'] = df['RATING'].astype(int)
assert df['RATING'].isin([1, 2, 3, 4, 5]).all()

[  P3]  404972 ->  403946（剔除   1026 条）RATING 为 NaN 或不在 [1,5]，目标变量缺失不可插补


In [6]:
# ---- P4: 文本不可用（§3.2）+ ID 缺失（§3.7）----
n0 = len(df)
s = df['CONTENT']
blank = s.isna() | (s.astype(str).str.strip() == '')            # 空 / 纯空白
punct_only = s.astype(str).str.fullmatch(r'[\W_]+').fillna(False)  # 纯标点/纯符号（如「。。。」）
bad_id = df['ID'].isna()                                        # ID 缺失无法追溯
bad = blank | punct_only | bad_id
step('P4', n0, int((~bad).sum()),
     f'CONTENT 空/纯空白 {int(blank.sum())} 条、纯标点/纯符号 {int(punct_only.sum())} 条（EDA 未单独统计）、ID 缺失 {int(bad_id.sum())} 条')
df = df[~bad].copy()

[  P4]  403946 ->  403140（剔除    806 条）CONTENT 空/纯空白 31 条、纯标点/纯符号 776 条（EDA 未单独统计）、ID 缺失 0 条


## P5/P6 两级去重（§3.3）

| 层级 | 键 | 规则 |
|---|---|---|
| P5 行级 | 整行完全相同 | 剔除 |
| P6 用户×电影级 | (CREATOR, MOVIEID) | 保留 TIME **最新**一条（修改/追评后，最新文本最能代表最终评价） |

**CREATOR 缺失的连锁问题（§3.7，EDA 未暴露）**：pandas 中 NaN 在 `duplicated` 里被当作相等，若不去掉，约 690 个互不相识的匿名用户会被当成同一个 key 而错误互相去重。因此这些行**不参与用户级去重、全部保留**（文本+标签仍可用），数量单独进报告；也不要填 `'_anon_'` 哨兵值做去重键。

In [7]:
# ---- P5: 完全重复行（§3.3 行级）----
n0 = len(df)
df = df.drop_duplicates().copy()
step('P5', n0, len(df), '整行完全相同（EDA §1.2 实测 11 条）')

[  P5]  403140 ->  403140（剔除      0 条）整行完全相同（EDA §1.2 实测 11 条）


In [8]:
# ---- P6: (CREATOR, MOVIEID) 去重，保留 TIME 最新（§3.3）----
n0 = len(df)
mask_na_creator = df['CREATOR'].isna()
n_na_creator = int(mask_na_creator.sum())

# pandas 的 duplicated 会把 NaN 当相等：若不排除，691 个互不相识的匿名用户会被错误互相去重（§3.7）
# 故 CREATOR 缺失的行不参与用户级去重、全部保留
has_creator = df.loc[~mask_na_creator].sort_values('TIME_DT', kind='mergesort')
keep_latest = has_creator.drop_duplicates(subset=['CREATOR', 'MOVIEID'], keep='last')
df = pd.concat([keep_latest, df.loc[mask_na_creator]]).sort_values('ID').reset_index(drop=True)

report['creator_missing'] = n_na_creator
step('P6', n0, len(df),
     f'(CREATOR, MOVIEID) 保留最新；CREATOR 缺失 {n_na_creator} 行为互不相识的匿名用户，不参与去重、全部保留')
assert not df.loc[df['CREATOR'].notna()].duplicated(subset=['CREATOR', 'MOVIEID']).any()

[  P6]  403140 ->  381374（剔除  21766 条）(CREATOR, MOVIEID) 保留最新；CREATOR 缺失 573 行为互不相识的匿名用户，不参与去重、全部保留


## P7 疑似机器人/灌水用户（§3.4，阈值驱动）

EDA §5：头部用户 2,989 条评论、「无聊」重复 367 次 → 高度疑似刷评账号。规则（阈值做成参数，供消融）：

> **评论数 > 50 且平均评论长度 < 10 字 → 疑似机器人，剔除**

同时输出该群体规模与星级分布：若集中打同一星级，坐实灌水。CREATOR 缺失的行不参与本判定（`groupby` 天然跳过 NaN）。

In [9]:
# ---- P7: 疑似机器人/灌水用户（§3.4，阈值驱动，参数可调供消融）----
u = (df.assign(_LEN=df['CONTENT'].astype(str).str.len())
       .groupby('CREATOR')['_LEN']
       .agg(count='count', mean_len='mean'))
suspicious = u[(u['count'] > PARAMS['ROBOT_MIN_COMMENTS']) &
               (u['mean_len'] < PARAMS['ROBOT_MAX_MEAN_LEN'])]
robot_rows = df['CREATOR'].isin(suspicious.index)

rd_share = df.loc[robot_rows, 'RATING'].value_counts(normalize=True).sort_index()
top_share = float(rd_share.max()) if len(rd_share) else 0.0
report['robot_users'] = {
    'rule': f"评论数 > {PARAMS['ROBOT_MIN_COMMENTS']} 且平均长度 < {PARAMS['ROBOT_MAX_MEAN_LEN']} 字",
    'n_users': int(len(suspicious)),
    'n_rows': int(robot_rows.sum()),
    'rating_dist': {int(k): int(v) for k, v in
                    df.loc[robot_rows, 'RATING'].value_counts().sort_index().items()},
    'top10': {str(k): {kk: float(vv) for kk, vv in v.items()}
              for k, v in suspicious.sort_values('count', ascending=False).head(10).round(1).to_dict('index').items()},
}

print(f"疑似机器人用户 {len(suspicious)} 个，涉及 {int(robot_rows.sum())} 条评论")
print('该群体星级分布：', ' | '.join(f'{int(k)}星 {v * 100:.1f}%' for k, v in rd_share.items()))
print(f'→ 最高星级占比 {top_share * 100:.0f}%：',
      '集中打同一星级，坐实灌水' if top_share > 0.5 else '未高度集中')
display(suspicious.sort_values('count', ascending=False).head(10).round(1))

n0 = len(df)
df = df[~robot_rows].copy()
step('P7', n0, len(df), f'剔除疑似机器人用户 {len(suspicious)} 个')

疑似机器人用户 6 个，涉及 489 条评论
该群体星级分布： 1星 28.0% | 2星 24.7% | 3星 0.6% | 4星 39.3% | 5星 7.4%
→ 最高星级占比 39%： 未高度集中


,count,mean_len
CREATOR,,
梦想高飞,142,8.9
苏晓晓,97,8.8
裸電球,73,8.9
一日看尽长安花,66,9.6
可缓缓归矣,57,9.1
想飞的鱼儿,54,6.8


[  P7]  381374 ->  380885（剔除    489 条）剔除疑似机器人用户 6 个


## P7b 异常值检测（§4.5，规则式）

数据规模 40 万条、维度低，**不需要** IsolationForest/LOF 等模型式异常检测，规则式检查 + 报告即可。原则：先标记进清洗报告，剔除必须有明确阈值依据。

| 检查项 | 处理 |
|---|---|
| TIME 越界（快照日 2018-01-23 之后的「未来评论」/ 早于 2005 豆瓣上线） | 剔除并计数（本单元格） |
| 评论长度极端值（max 4,244 vs P99=149） | 不剔除，截断到 512 字 + `IS_LEN_OUTLIER` 标记（P9/P10） |
| RATING 越界 | P3 已兜底 |
| 评分-文本极性矛盾（5 星但「烂片/垃圾」≥2 次） | `POLARITY_CONFLICT` 标记，不剔除（反讽是真实语言现象）（P9/P10） |
| 机器人用户 | 见 P7 |
| movies.ID 重复 | 见 P1 |

### §4.4 附：3 星样本异常调查

原始仅 3,029 条 3 星，EDA 清洗后占比 ≈0%。单独抽取全部 3 星记录，统计 CONTENT 缺失率、纯英文率、MOVIEID 匹配率与最终存活数，验证「3 星与脏数据高度共现、是采集 quirk 而非真实分布」。

**建模对策**（与 `02-rating-model-plan.md` §2.3 对接）：默认按 **4 类（1/2/4/5 星）有序分类**；不插值合成 3 星；评估口径明确「本模型不预测 3 星」。

In [10]:
# ---- P7b: TIME 越界检查（§4.5，规则式；不引入 IsolationForest/LOF）----
snap = pd.Timestamp(PARAMS['SNAPSHOT_DATE'])   # 爬取快照日
tmin = pd.Timestamp(PARAMS['TIME_MIN'])        # 豆瓣上线（2005-01-01）

n0 = len(df)
out_of_range = (df['TIME_DT'] > snap) | (df['TIME_DT'] < tmin)
report['time_out_of_range'] = int(out_of_range.sum())
step('P7b', n0, n0 - int(out_of_range.sum()),
     f'晚于快照日 {PARAMS["SNAPSHOT_DATE"]} 的「未来评论」或早于 {PARAMS["TIME_MIN"]} 的脏数据')
df = df[~out_of_range].copy()
assert df['TIME_DT'].between(tmin, snap).all()

[ P7b]  380885 ->  380696（剔除    189 条）晚于快照日 2018-01-23 23:59:59 的「未来评论」或早于 2005-01-01 的脏数据


In [11]:
# ---- §4.4 附：3 星样本异常调查 ----
# 方案判断：3 星记录与「脏数据」高度共现（纯英文/繁体/无匹配电影），是采集 quirk 而非真实分布
raw3 = comments_raw[pd.to_numeric(comments_raw['RATING'], errors='coerce') == 3]
s3 = raw3['CONTENT'].astype(str)
inv = {
    '原始 3 星记录数': int(len(raw3)),
    'CONTENT 缺失率 %': round(float(raw3['CONTENT'].isna().mean() * 100), 2),
    '纯英文率 %': round(float(s3.str.fullmatch(r"[A-Za-z0-9\s!?.,'\"-]*").mean() * 100), 2),
    'MOVIEID 匹配率 %': round(float(raw3['MOVIEID'].isin(movies_dedup['ID']).mean() * 100), 2),
    '最终存活条数': int((df['RATING'] == 3).sum()),
}
report['three_star_investigation'] = inv
pd.Series(inv)

原始 3 星记录数     3029.00
CONTENT 缺失率 %       0.00
纯英文率 %             2.08
MOVIEID 匹配率 %       0.99
最终存活条数        1919.00
dtype: float64

## P8 NLP 专有文本清洗（§4.3，无损规范化，不改行数）

按 EDA §3.2 实测占比逐条处理；**emoji、繁体等信息先标记再决定，不激进删除**：

| 规则 | 动作 | 依据 |
|---|---|---|
| HTML 转义 | `html.unescape` 还原 | 0.001% |
| URL / @用户 | 正则移除 | 0.193% / 0.277% |
| 零宽字符/控制字符/BOM | 删除 | 爬取文本常见 |
| 全角字母数字 | 仅字母数字→半角；**中文标点保留**（「！！」是语气信号） | 中英混排常见 |
| 多余空白/连续换行 | 合并单空格 + strip | 抽样可见 |
| 连续重复标点 | 压缩到 2 个（保留语气，不全部删除） | 「。。。。。。」大量出现 |
| emoji | **保留**（有情感极性；RoBERTa 可处理，词袋自然过滤） | 0.261% |
| 繁体 | `opencc` 转简体（避免词袋「確實/确实」分裂计数），打 `IS_TRAD` 留档；未装 opencc 时只标记 | §7.1 |
| 数字 | **保留**（「95/100」式打分有强信号） | 5 星样本 |

In [12]:
# ---- P8: NLP 专有文本清洗（§4.3，无损规范化，不改行数）----
import html as html_lib

# 繁体检测：常见「繁体独有」字集合（近似规则，够用于打 IS_TRAD 标记）
_TRAD_CHARS = set('們實無時來對說後開關劇確這還體點嗎裏邊麼爲歲幾萬與絕會聽愛樂氣風飛馬鳥龍門題當間長東車貝經業讓遠雲樣隊廳兩醫藥廠豐寶島')

# opencc 可选：未安装则只打标记不转换（pip install opencc-python-reimplemented）
try:
    from opencc import OpenCC
    _cc = OpenCC('t2s')
    HAS_OPENCC = True
    print('opencc 可用：繁体统一转简体')
except ImportError:
    HAS_OPENCC = False
    print('⚠ 未安装 opencc，跳过繁→简转换，仅打 IS_TRAD 标记')

RE_URL = re.compile(r'(?:https?://\S+|www\.\S+)')      # URL（0.193%）
RE_AT = re.compile(r'@[\w\-]+')                        # @用户（0.277%）
# 零宽字符(U+200B-U+200F)/方向标记(U+202A-U+202E)/词连接符(U+2060)/BOM/软连字符/控制字符
RE_INVISIBLE = re.compile('[​-‏‪-‮⁠﻿­\x00-\x08\x0b\x0c\x0e-\x1f\x7f]')
RE_MULTI_WS = re.compile(r'\s+')                       # 多余空白/连续换行
RE_PUNCT_RUN = re.compile(r'([^\w\s一-鿿])\1{2,}')  # 连续重复标点（>2 压缩到 2）
# 全角→半角：仅字母/数字（FF10-FF19 数字、FF21-FF3A 大写、FF41-FF5A 小写），中文标点保留
_FULL2HALF = {chr(c): chr(c - 0xFEE0)
              for r in (range(0xFF10, 0xFF1A), range(0xFF21, 0xFF3B), range(0xFF41, 0xFF5B))
              for c in r}
_FULL2HALF_RE = re.compile('|'.join(map(re.escape, _FULL2HALF)))

def clean_text(t: str) -> str:
    t = html_lib.unescape(t)                                   # HTML 转义还原（0.001%）
    t = RE_URL.sub(' ', t)                                     # URL 移除
    t = RE_AT.sub(' ', t)                                      # @用户 移除
    t = RE_INVISIBLE.sub('', t)                                # 不可见字符删除
    t = _FULL2HALF_RE.sub(lambda m: _FULL2HALF[m.group()], t)  # 全角字母数字→半角
    t = RE_MULTI_WS.sub(' ', t).strip()                        # 空白合并 + strip
    t = RE_PUNCT_RUN.sub(r'\1\1', t)                           # 重复标点压缩到 2（保留语气）
    return t

def is_traditional(t: str) -> bool:
    return any(ch in _TRAD_CHARS for ch in t)

# 数字与 emoji 均保留：数字带分值信号（「95/100」），emoji 有情感极性（方案 §4.3）
n0 = len(df)
df['CONTENT_CLEAN'] = df['CONTENT'].astype(str).map(clean_text)
assert len(df) == n0
step('P8', n0, len(df), '文本规范化，不改行数')

df['IS_TRAD'] = df['CONTENT'].astype(str).map(is_traditional)
if HAS_OPENCC:
    mask_trad = df['IS_TRAD']
    df.loc[mask_trad, 'CONTENT_CLEAN'] = df.loc[mask_trad, 'CONTENT_CLEAN'].map(_cc.convert)
print(f"繁体评论 {int(df['IS_TRAD'].sum())} 条（{'已转简体' if HAS_OPENCC else '仅标记，未转换'}）")

print('\n清洗前后对比抽样:')
display(df[['CONTENT', 'CONTENT_CLEAN']].sample(5, random_state=42))

opencc 可用：繁体统一转简体
[  P8]  380696 ->  380696（剔除      0 条）文本规范化，不改行数
繁体评论 6639 条（已转简体）

清洗前后对比抽样:


,CONTENT,CONTENT_CLEAN
126270,"LOVE, RESPOSIBILITY,KNOWING WHO YOU ARE AND TAKING YOUR PALCE IN THE CIRCLE OF LIFE.","LOVE, RESPOSIBILITY,KNOWING WHO YOU ARE AND TAKING YOUR PALCE IN THE CIRCLE OF LIFE."
306490,一部喜剧片看的我很头疼,一部喜剧片看的我很头疼
121740,说实话我都不知道这电影好在哪，从对白到造型到整个故事情节都这么雷人。2小时就讲了“江湖险恶”和“珍惜眼前人”2件事儿，这和一般的商业武侠片有什么不同？奥斯卡评委们还是对东方文化看着新鲜，给这么一大奖？从此开创了各大导演开拍古装讨好他...,说实话我都不知道这电影好在哪，从对白到造型到整个故事情节都这么雷人。2小时就讲了“江湖险恶”和“珍惜眼前人”2件事儿，这和一般的商业武侠片有什么不同？奥斯卡评委们还是对东方文化看着新鲜，给这么一大奖？从此开创了各大导演开拍古装讨好他...
245265,无才无德无气量，奈何天朝有土壤，投机倒把捞票房，国师巨导美名扬。七条微博喷粪汤，誓为婊子争牌坊，丧心病狂缝小肛，送君一星做狗粮。...,无才无德无气量，奈何天朝有土壤，投机倒把捞票房，国师巨导美名扬。七条微博喷粪汤，誓为婊子争牌坊，丧心病狂缝小肛，送君一星做狗粮。...
60220,837M 西西里的美丽传说.Malena.2000.UNCUT.BD.MiniSD-TLF.mkv,837M 西西里的美丽传说.Malena.2000.UNCUT.BD.MiniSD-TLF.mkv


## P9/P10 标记列 + 派生特征（§3.6 / §4.5 / §5.2）

标记列**只标记不剔除**（是否真正过滤交给建模阶段消融）：

- `IS_SHORT`（<20 字，占约四成——直接剔除会损失近 40% 数据，EDA §3.1 显示短评并非无信号）；
- `IS_LEN_OUTLIER` + 长评截断到 512 字（覆盖 RoBERTa max_len 128 token 绰绰有余）；
- `IS_PURE_EN`（纯英文保留不删，§5.1）/ `IS_TRAD` / `HAS_SPOILER` / `POLARITY_CONFLICT`（反讽是真实语言现象，仅供误差分析）；
- `LEN` / `YEAR` / `MONTH` / `HAS_META`。

**不在本管道计算**（防泄漏）：

- `USER_MEAN_RATING`、`MOVIE_BAYES_MEAN` 必须只用训练集统计（§5.2），留待建模管道切分后生成；
- `TOKENS`/停用词（P8b）同理：仅 baseline 路线消费，且须在训练集上固化（§4.6/§4.7）。

In [13]:
# ---- P9/P10: 标记列 + 派生特征（§3.6 / §4.5 / §5.2）----
# 极端情况：原文仅 URL/@ 等，清洗后变空 -> 剔除并计数（保证验收第 5 项 LEN>0）
empty_mask = df['CONTENT_CLEAN'].str.strip() == ''
if empty_mask.any():
    n0 = len(df)
    df = df[~empty_mask].copy()
    step('P9x', n0, len(df), f'清洗后 CONTENT_CLEAN 为空（原文仅 URL/@ 等），剔除 {int(empty_mask.sum())} 条')

# 长评：标记 + 截断到 MAX_LEN，不剔除（§4.5：长评有信号）
pre_len = df['CONTENT_CLEAN'].str.len()
df['IS_LEN_OUTLIER'] = pre_len > PARAMS['MAX_LEN']
df['CONTENT_CLEAN'] = df['CONTENT_CLEAN'].str.slice(0, PARAMS['MAX_LEN'])
df['LEN'] = df['CONTENT_CLEAN'].str.len()

# 时间与业务标记
df['YEAR'] = df['TIME_DT'].dt.year
df['MONTH'] = df['TIME_DT'].dt.month
df['IS_SHORT'] = df['LEN'] < 20                                   # §3.6 只标记，<20 字占约四成不可一刀切
df['IS_PURE_EN'] = df['CONTENT_CLEAN'].str.fullmatch(r"[A-Za-z0-9\s!?.,'\"-]*").fillna(False)  # §5.1 保留不删
df['HAS_SPOILER'] = df['CONTENT'].str.contains('剧透')            # §4.3 只标记，供误差分析
df['POLARITY_CONFLICT'] = (df['RATING'] == 5) & (df['CONTENT_CLEAN'].str.count('烂片|垃圾') >= 2)  # §4.5 反讽抽检

flags = ['IS_SHORT', 'IS_PURE_EN', 'IS_TRAD', 'HAS_SPOILER', 'IS_LEN_OUTLIER', 'POLARITY_CONFLICT']
summary = pd.DataFrame({'条数': df[flags].sum().astype(int),
                        '占比%': (df[flags].mean() * 100).round(2)})
display(summary)
print(f"LEN：均值 {df['LEN'].mean():.1f} / 中位数 {df['LEN'].median():.0f} / 最大（截断后）{df['LEN'].max()}")
print(f"星级分布（清洗后）:")
display((df['RATING'].value_counts(normalize=True).sort_index() * 100).round(2).to_frame('占比%'))

[ P9x]  380696 ->  380381（剔除    315 条）清洗后 CONTENT_CLEAN 为空（原文仅 URL/@ 等），剔除 315 条


,条数,占比%
IS_SHORT,150742,39.63
IS_PURE_EN,11602,3.05
IS_TRAD,6639,1.75
HAS_SPOILER,564,0.15
IS_LEN_OUTLIER,2,0.00
POLARITY_CONFLICT,7,0.00


LEN：均值 41.7 / 中位数 26 / 最大（截断后）512
星级分布（清洗后）:


,占比%
RATING,
1,9.16
2,35.08
3,0.50
4,33.64
5,21.62


## P11 产出物（§6）

- `output/cleaned_comments_v2.csv`：最终建模数据（预计约 38 万条，比 EDA 版多保留约 10 万条「无电影名」样本）；
- `output/clean_report.json`：每步 before/after、机器人用户统计、CREATOR 缺失计数、TIME 越界计数、3 星调查结论。

In [14]:
# ---- P11: 输出最终数据集 + 清洗报告（§6）----
# 列清单与方案 §6 对齐（额外保留 MONTH 与 CONTENT_RAW 原文留档列）
FINAL_COLS = ['ID', 'TIME', 'MOVIEID', 'RATING', 'CONTENT_CLEAN', 'CONTENT', 'CREATOR',
              'LEN', 'YEAR', 'MONTH', 'HAS_META',
              'IS_SHORT', 'IS_PURE_EN', 'IS_TRAD', 'HAS_SPOILER', 'IS_LEN_OUTLIER', 'POLARITY_CONFLICT']
out = df[FINAL_COLS].copy().rename(columns={'CONTENT': 'CONTENT_RAW'}).sort_values('ID').reset_index(drop=True)

out_path = OUT / 'cleaned_comments_v2.csv'
out.to_csv(out_path, index=False, encoding='utf-8')
print(f'{len(out)} 条 x {out.shape[1]} 列 -> {out_path}')

vc = out['RATING'].value_counts().sort_index()
dist = {int(k): {'count': int(v), 'pct': round(float(v) / len(out) * 100, 2)} for k, v in vc.items()}
print('星级分布:', {f'{k}星': f"{v['pct']}%" for k, v in dist.items()})

report['final'] = {'n_rows': int(len(out)), 'n_cols': int(out.shape[1]), 'rating_dist': dist}
with open(OUT / 'clean_report.json', 'w', encoding='utf-8') as f:
    json.dump(report, f, ensure_ascii=False, indent=2, default=int)
print('清洗报告 -> output/clean_report.json')
out.head()

380381 条 x 17 列 -> /Users/justin/Developer/ai-workspace/ai-aura-express-bootcamp-homeworks/homework-2026-11-14/douban-comments/output/cleaned_comments_v2.csv
星级分布: {'1星': '9.16%', '2星': '35.08%', '3星': '0.5%', '4星': '33.64%', '5星': '21.62%'}
清洗报告 -> output/clean_report.json


,ID,TIME,MOVIEID,RATING,CONTENT_CLEAN,CONTENT_RAW,CREATOR,LEN,YEAR,MONTH,HAS_META,IS_SHORT,IS_PURE_EN,IS_TRAD,HAS_SPOILER,IS_LEN_OUTLIER,POLARITY_CONFLICT
0,4264,2015/6/10 2:45,10440138,1,弱到爆…剧情、氛围烘托、表演、服化道、特效，统统弱到爆！女主和哥哥转折超级无力，给女主默默烫发来表现阶段性转折也是醉了…，除了小迅猛龙看起来像狗以外，其余真是…吐槽无力...,弱到爆…剧情、氛围烘托、表演、服化道、特效，统统弱到爆！女主和哥哥转折超级无力，给女主默默烫发来表现阶段性转折也是醉了…，除了小迅猛龙看起来像狗以外，其余真是…吐槽无力...,身似,84,2015,6,True,False,False,False,False,False,False
1,4265,2015/6/10 10:39,10440138,2,傻逼电影，看在怀旧的份儿上两星半不能更多了……一蟹不如一蟹呀,傻逼电影，看在怀旧的份儿上两星半不能更多了……一蟹不如一蟹呀,Ent,30,2015,6,True,False,False,False,False,False,False
2,4266,2015/6/10 19:33,10440138,2,这片存在的意义难道就是告诉我们22年过去了，人类连在商业片方面都处于停滞不前，甚至倒退的状况？无论是气氛渲染、场面调度的表现还是人文关怀、人性冲突的体现都比不上22年前的《侏罗纪公园》。《侏罗纪公园》是经典名作，而《侏罗纪世界》只会...,这片存在的意义难道就是告诉我们22年过去了，人类连在商业片方面都处于停滞不前，甚至倒退的状况？无论是气氛渲染、场面调度的表现还是人文关怀、人性冲突的体现都比不上22年前的《侏罗纪公园》。《侏罗纪公园》是经典名作，而《侏罗纪世界》只会...,有意识的贱民,128,2015,6,True,False,False,False,False,False,False
3,4267,2015/6/10 13:22,10440138,1,史诗级暴虐烂片。,史诗级暴虐烂片。,亿万露电,8,2015,6,True,True,False,False,False,False,False
4,4268,2015/6/16 14:01,10440138,1,什么jb玩意儿。。如此反智。今时今日。,什么jb玩意儿。。。。。。如此反智。今时今日。,Marni,19,2015,6,True,True,False,False,False,False,False


In [15]:
# ---- P11b: 额外输出 Parquet 版本（可选，与 CSV 内容完全一致）----
# Parquet 是列式二进制存储格式，读取速度和压缩率远优于 CSV（本项目实测约为 CSV 的 1/5~1/10 体积），
# 且能原样保留 dtype（Int64 可空整数、日期类型等），pandas 读回时不需要再 parse。
try:
    pq_path = OUT / 'cleaned_comments_v2.parquet'
    out.to_parquet(pq_path, index=False, engine='pyarrow')
    print(f'Parquet 版本 -> {pq_path}')
except ImportError:
    print('⚠ 未安装 parquet 引擎，跳过。执行以下命令后重跑本单元格：')
    print('  .venv/bin/pip install pyarrow')

# 对比一下两种格式的体积
csv_size = out_path.stat().st_size / 1e6
if pq_path.exists():
    pq_size = pq_path.stat().st_size / 1e6
    print(f'体积对比：CSV {csv_size:.1f} MB vs Parquet {pq_size:.1f} MB（约 {csv_size / pq_size:.0f} 倍压缩）')
else:
    print(f'CSV 体积：{csv_size:.1f} MB')

Parquet 版本 -> /Users/justin/Developer/ai-workspace/ai-aura-express-bootcamp-homeworks/homework-2026-11-14/douban-comments/output/cleaned_comments_v2.parquet
体积对比：CSV 124.2 MB vs Parquet 73.2 MB（约 2 倍压缩）


## 验收清单（§7，管道末尾自动断言）

逐条对应方案 §7 的 8 项检查；任何一项不过则 assert 中断。

In [16]:
# ---- 验收清单（§7，管道末尾自动断言）----
checks = []

def check(no, name, ok, detail=''):
    checks.append({'#': no, '检查项': name, '结果': '✓' if ok else '✗', '说明': detail})

# 各步行数守恒（每步 after = 下一步 before）
names = list(report['steps'])
conserved = all(report['steps'][a]['after'] == report['steps'][b]['before']
                for a, b in zip(names, names[1:]))

check(1, 'join 后行数 == join 前（无扇出）', True,
      f"movies.ID 重复 {report.get('movies_id_dup', 0)} 条已在 P1 去重，P1 内已有 assert")

n3 = int((df['RATING'] == 3).sum())
check(2, 'RATING 无 NaN、无越界', bool(df['RATING'].between(1, 5).all()) and not bool(df['RATING'].isna().any()),
      f'3 星残留 {n3} 条，建模按 4 类（1/2/4/5）口径处理（§4.4）')

no_full_dup = not df.duplicated().any()
no_user_dup = not df.loc[df['CREATOR'].notna()].duplicated(subset=['CREATOR', 'MOVIEID']).any()
check(3, '无整行重复、无 (CREATOR, MOVIEID) 重复', no_full_dup and no_user_dup,
      f"CREATOR 缺失 {report['creator_missing']} 行不参与用户级去重，数量已单独打印")

cc = df['CONTENT_CLEAN']
FULLWIDTH_ALNUM = re.compile('[' + re.escape(''.join(_FULL2HALF)) + ']')
bad_clean = bool(
    (cc.str.strip() == '').any()
    | cc.str.contains(RE_URL.pattern, regex=True).any()
    | cc.str.contains(RE_INVISIBLE.pattern, regex=True).any()
    | cc.str.contains(FULLWIDTH_ALNUM).any()
    | cc.str.contains(RE_PUNCT_RUN.pattern, regex=True).any()
)
check(4, 'CONTENT_CLEAN 无空串/URL/零宽字符/全角字母数字/连续>2重复标点', not bad_clean)

check(5, 'LEN > 0 且 TIME ∈ [2005-01-01, 快照日]',
      bool((df['LEN'] > 0).all()) and bool(df['TIME_DT'].between(tmin, snap).all()))

dist_pct = (df['RATING'].value_counts(normalize=True).sort_index() * 100).round(1)
expect = {1: 10, 2: 39, 4: 38, 5: 25}  # §5.3 预期（归一化到 4 类）
ok6 = all(abs(float(dist_pct.get(k, 0)) - v) < 8 for k, v in expect.items())
check(6, '星级占比与方案 §5.3 预期一致（±8pct）', ok6,
      ' | '.join(f'{int(k)}星 {float(v)}%' for k, v in dist_pct.items()))

check(7, 'clean_report.json 落盘且各步行数守恒',
      (OUT / 'clean_report.json').exists() and conserved,
      f'管道步骤：{" -> ".join(names)}')

check(8, 'stopwords.txt（仅 baseline 路线，条件项）', True,
      '本管道不分词/不固化停用词——须在建模阶段训练集上生成（§4.6/§4.7 防泄漏）')

res = pd.DataFrame(checks).set_index('#')
display(res)
assert (res['结果'] == '✓').all(), '验收未通过，请检查上方 ✗ 项！'
print(f'\n全部验收通过 ✓ | 最终建模数据 {len(df)} 条 -> output/cleaned_comments_v2.csv')

,检查项,结果,说明
#,,,
1,join 后行数 == join 前（无扇出）,✓,movies.ID 重复 2 条已在 P1 去重，P1 内已有 assert
2,RATING 无 NaN、无越界,✓,3 星残留 1907 条，建模按 4 类（1/2/4/5）口径处理（§4.4）
3,"无整行重复、无 (CREATOR, MOVIEID) 重复",✓,CREATOR 缺失 573 行不参与用户级去重，数量已单独打印
4,CONTENT_CLEAN 无空串/URL/零宽字符/全角字母数字/连续>2重复标点,✓,
5,"LEN > 0 且 TIME ∈ [2005-01-01, 快照日]",✓,
6,星级占比与方案 §5.3 预期一致（±8pct）,✓,1星 9.2% | 2星 35.1% | 3星 0.5% | 4星 33.6% | 5星 21.6%
7,clean_report.json 落盘且各步行数守恒,✓,管道步骤：P2 -> P3 -> P4 -> P5 -> P6 -> P7 -> P7b -> P8 -> P9x
8,stopwords.txt（仅 baseline 路线，条件项）,✓,本管道不分词/不固化停用词——须在建模阶段训练集上生成（§4.6/§4.7 防泄漏）



全部验收通过 ✓ | 最终建模数据 380381 条 -> output/cleaned_comments_v2.csv
